# Monte-Carlo tree search: Trees and search

Python / Numpy implementation & comparison against faster reference implementation ([MCTX](https://github.com/google-deepmind/mctx)), in partuclar the [visualization demo](https://github.com/google-deepmind/mctx/blob/main/examples/visualization_demo.py) example of MCTX.

Known limitations:
* Seeds are not fully in line with MCTX. Small random noise is used for tie breaking in arg_max over q-values for action selection.
* No action masking implemented yet.
* No Dirichlet noise implemented yet.

Relevant parts of the MuZero paper:
* MuZero only masks legal actions at the root of the search tree where the environment can be queried,
* MuZero does not give special treatment to terminal nodes and always uses the value predicted by the network. 
  Inside the tree, the search can proceed past a terminal node - in this case the network is expected 
  to always predict the same value. This is achieved by treating terminal states as absorbing states during training.
* Every node of the search tree is associated with an internal state s. For each action a from s there is an
  edge (s, a) that stores a set of statistics {N(s, a), Q(s, a), P (s, a), R(s, a), S(s, a)}, respectively 
  representing visit counts N, mean value Q, policy P, reward R, and state transition S.

In [5]:
import numpy as np

import pygraphviz as pgv

import jax # only to track seeds to add noise to break ties in argmax, as the MCTX example does

import sys, os
sys.path.append(os.getcwd()+"/../src")

from mcts import MCTSNode, MCTSTree, draw_tree

In [6]:
seed = 42
rng_key = jax.random.PRNGKey(seed)

def g_mctx(s, a):
    transition_matrix = np.array([
        [1, 2, 3, 4],
        [0, 5, 0, 0],
        [0, 0, 0, 6],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
    ], dtype=np.int32)
    rewards = np.array([
        [1, -1, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [10, 0, 20, 0],
    ], dtype=np.float32)
    r = rewards[s, a]
    s_next = transition_matrix[s, a]
    return r, s_next

def f_mctx(s):
    v = 15.0
    n_actions = 4
    p=np.ones(n_actions)/n_actions
    return p, v

tree = MCTSTree(
    MCTSNode(
        hidden_state=0, # discrete states
        n_actions=4,
        v = 15.0,
        identifier="root",
        ),
    n_simulations=10, # fails for n_simulations = 12
    dynamics_function=g_mctx,
    prediction_function=f_mctx,
    n_actions=4,
    rng_key=rng_key
)
tree.search()
draw_tree(tree, out_file="../output/reproduction_mctx_visualization_demo.png")

Original MCTX visualization demo (seed 42, 10 simulations):

![](../assets/mctx_visualization_demo.png)

My reproduction:

![](../assets/reproduction_mctx_visualization_demo.png)